In [ ]:
using Yao
using Optim
using LinearAlgebra
using StatsBase
using Printf

# Problema de la Mochila en Digital

Habiendo implementado el problema de la mochila en computación analógica, hemos topado con límites cuando el número $N$ de activos pasa de las decenas y empieza a haber problemas para mapear el sistema en un grafo planar. Es por esto que vamos a dar el salto a la computación digital, ahora la correlación de los activos no será un problema, ya que se implementarán a nuestro problema mediante puertas $CZ$. Implementaremos el algoritmo QAOA (Quantum Approximate Optimization Algorithm), que evita el problema de la planaridad de los grafos.

## El Toy-Model (N=4, K=2)

Comenzaremos con el caso sencillo de cuatro activos para asentar la teoría:

*  **Activo 1:** Retorno alto (8%), pero muy volátil (riesgo propio = 10).
*  **Activo 2:** Retorno bueno (7%), riesgo medio (5).
*  **Activo 3:** Retorno medio (5%), riesgo medio (6). Muy correlacionado con el **activo 1**.
*  **Activo 4:** Retorno bajo (3%), riesgo casi nulo (2).

$$\mu = (8, 7, 5, 3), \qquad \Sigma = \begin{pmatrix} 10 & 2 & 8 & 1 \\ 2 & 5 & 3 & 0 \\ 8 & 3 & 6 & 1 \\ 1 & 0 & 1 & 2 \end{pmatrix}$$

In [ ]:
# Vector de retornos y matriz de covarianza del Toy-Model
μ_toy  = [8.0, 7.0, 5.0, 3.0]
N_toy  = length(μ_toy)
sigma_toy = [10.0 2.0 8.0 1.0;
              2.0 5.0 3.0 0.0;
              8.0 3.0 6.0 1.0;
              1.0 0.0 1.0 2.0];

## Transformación QUBO

Para pasar a variables binarias $x_i \in \{0,1\}$ que representen si compramos o no un activo, definimos la función de coste:
$$H = -\underbrace{\sum_i \mu_i x_i}_{\text{Retorno}} + \gamma \underbrace{\sum_{i,j} x_i \Sigma_{ij} x_j}_{\text{Riesgo}} + \lambda \underbrace{\left( \sum_i x_i - K \right)^2}_{\text{Restricción}}$$

Aplicando $x_i^2 = x_i$ obtenemos la matriz QUBO con coeficientes exactos:

$$Q_{ii} = -\mu_i + \gamma \Sigma_{ii} + \lambda(1 - 2K), \qquad Q_{ij} = 2\gamma\Sigma_{ij} + 2\lambda \quad (i \neq j)$$

In [ ]:
# Función genérica para construir Q (vectorizada, sin bucle doble)
function construir_Q(μ, sigma, K; γ=1.0, λ=5.0)
    N = length(μ)
    Q = 2γ .* sigma .+ 2λ                                  # off-diagonal
    for i in 1:N
        Q[i, i] = -μ[i] + γ * sigma[i, i] + λ * (1 - 2K)  # diagonal
    end
    Q ./= maximum(abs.(Q))                                  # normalizar a [-1,1]
    return Q
end

K_toy = 2
Q_toy = construir_Q(μ_toy, sigma_toy, K_toy; γ=1.0, λ=5.0)
println("Matriz Q (Toy-Model):")
display(round.(Q_toy, digits=3))

## Implementación del QAOA Estándar

### Hamiltoniano de Coste

Mapeando los bits clásicos $x_i \in \{0,1\}$ a espines cuánticos mediante $x_i = \frac{I - Z_i}{2}$, la función de coste se convierte en el **Hamiltoniano de Ising**:

$$H_C = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$$

donde los coeficientes se obtienen directamente de Q:
$$h_i = -\frac{Q_{ii}}{2} - \sum_{j \neq i} \frac{Q_{ij}}{2}, \qquad J_{ij} = \frac{Q_{ij}}{2}$$

El operador de evolución $U_C(\gamma) = e^{-i\gamma H_C}$ se descompone en:
- Término lineal: puerta $R_z(2\gamma h_i)$ en el qubit $i$
- Término cuadrático: secuencia $\text{CNOT} - R_z(2\gamma J_{ij}) - \text{CNOT}$

### Hamiltoniano de Mezcla Estándar ($R_x$)

$$H_M = \sum_i X_i \implies U_M(\beta) = \bigotimes_i R_x(2\beta)$$

El estado inicial es $|\psi_0\rangle = H^{\otimes N}|0\rangle^{\otimes N} = \frac{1}{\sqrt{2^N}}\sum_x|x\rangle$, la superposición uniforme de todos los estados.

In [ ]:
# ─── QAOA Estándar (para el Toy-Model N=4) ────────────────────────────────────
function capa_QAOA_std(N, gamma, beta, Q)
    capa = chain(N)
    # Coeficientes de Ising
    h = [-Q[i,i]/2 - sum(Q[i,j]/2 for j in 1:N if j≠i) for i in 1:N]
    J = [(i<j) ? Q[i,j]/2 : 0.0 for i in 1:N, j in 1:N]

    # Hamiltoniano de coste
    for i in 1:N
        push!(capa, put(N, i => Rz(2gamma * h[i])))
        for j in (i+1):N
            if abs(Q[i,j]) > 1e-3
                push!(capa, cnot(N, i, j))
                push!(capa, put(N, j => Rz(2gamma * J[i,j])))
                push!(capa, cnot(N, i, j))
            end
        end
    end
    # Hamiltoniano de mezcla Rx
    for i in 1:N
        push!(capa, put(N, i => Rx(2beta)))
    end
    return capa
end

function circuito_QAOA_std(N, p, gammas, betas, Q)
    circ = chain(N)
    for i in 1:N
        push!(circ, put(N, i => H))  # Estado inicial |+>^N
    end
    for it in 1:p
        push!(circ, capa_QAOA_std(N, gammas[it], betas[it], Q))
    end
    return circ
end;

## Optimización del Toy-Model

Minimizamos el valor esperado de la energía:
$$E(\boldsymbol{\gamma}, \boldsymbol{\beta}) = \langle \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) | H_C | \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) \rangle$$

mediante gradiente descendente (L-BFGS). Las energías de los $2^N$ estados se **precalculan fuera del bucle** para evitar recomputarlas en cada evaluación.

In [ ]:
p_toy = 5

# Precalculamos las energías QUBO de los 2^N estados (fuera del hot-path)
E_prec_toy = [dot(digits(s, base=2, pad=N_toy), Q_toy * digits(s, base=2, pad=N_toy))
              for s in 0:(2^N_toy - 1)]

function perdida_toy(θ)
    gammas, betas = θ[1:p_toy], θ[p_toy+1:end]
    reg = zero_state(N_toy)
    apply!(reg, circuito_QAOA_std(N_toy, p_toy, gammas, betas, Q_toy))
    return real(dot(abs2.(statevec(reg)), E_prec_toy))
end

mejor_E_toy, mejores_θ_toy = Inf, zeros(2p_toy)
for _ in 1:50
    θ₀ = vcat(sort(rand(p_toy)) .* 2π, sort(rand(p_toy), rev=true))
    res = optimize(perdida_toy, θ₀, LBFGS(), Optim.Options(iterations=2500))
    if Optim.minimum(res) < mejor_E_toy
        mejor_E_toy   = Optim.minimum(res)
        mejores_θ_toy = Optim.minimizer(res)
    end
end
println("Energía mínima Toy-Model: ", round(mejor_E_toy, digits=4))

In [ ]:
reg_toy = zero_state(N_toy)
apply!(reg_toy, circuito_QAOA_std(N_toy, p_toy,
       mejores_θ_toy[1:p_toy], mejores_θ_toy[p_toy+1:end], Q_toy))
probs_toy = abs2.(statevec(reg_toy))

carteras_toy = sort([(digits(i, base=2, pad=N_toy), probs_toy[i+1])
                     for i in 0:(2^N_toy-1)], by=x->x[2], rev=true)

println("\n ─── TOP 3 CARTERAS (Toy-Model N=4, K=2) ───")
for k in 1:3
    bits, pk = carteras_toy[k]
    activos  = findall(==(1), bits)
    @printf("  %d.  %5.2f%%  →  Comprar: %s\n", k, pk*100, join(activos, ", "))
end

---
## Caso Realista: N=10, K=6

### El problema de escalar el QAOA estándar

Con N=4 se obtienen probabilidades cercanas al 99% en el estado óptimo. Al escalar a N=10 las probabilidades caen por debajo del 1%. Esto **no es un bug**, es una consecuencia física fundamental con dos causas:

**1. Dilución exponencial del espacio de Hilbert**

| | N=4, K=2 | N=10, K=6 |
|---|---|---|
| Estados totales | $2^4 = 16$ | $2^{10} = 1024$ |
| Estados válidos ($\sum x_i = K$) | $\binom{4}{2} = 6$ | $\binom{10}{6} = 210$ |
| Estados **inválidos** que combatir | 10 | **814** |
| Fracción del espacio válida | 37.5% | 20.5% |

**2. Barren Plateaus**

El estado inicial $|+\rangle^{\otimes N}$ distribuye amplitud uniformemente por los 1024 estados, incluyendo los 814 que violan la restricción K=6. El término de penalización $\lambda$ debe empujar toda esa probabilidad de vuelta a los 210 estados válidos usando solo las p=12 capas del circuito. El gradiente se vuelve exponencialmente pequeño en el paisaje de optimización (efecto *Barren Plateau*).

### La solución: QAOA con Restricción Dura

La clave es pasar de una restricción **suave** (penalización $\lambda$) a una restricción **dura** (garantizada por la estructura del circuito). Para ello usamos dos ingredientes:

#### 1. Estado de Dicke $|D_K^N\rangle$

En lugar del estado de Hadamard $|+\rangle^{\otimes N}$, preparamos el **Estado de Dicke**: la superposición uniforme de todos los estados con exactamente $K$ activos activados:

$$|D_K^N\rangle = \frac{1}{\sqrt{\binom{N}{K}}} \sum_{|x|=K} |x\rangle$$

Para N=10, K=6, esto nos da directamente 210 estados con amplitud igual, todos válidos. La penalización $\lambda$ **desaparece** de la función de coste.

La preparación eficiente usa el algoritmo de Bartschi-Eidenbenz (2019) con complejidad $O(NK)$:

#### 2. Mixer XY

El mixer estándar $R_x$ aplica un flip individual a cada qubit: $|0\rangle \leftrightarrow |1\rangle$. Esto **rompe la restricción K** porque puede cambiar el número de activos seleccionados. Para conservarlo, necesitamos un operador que intercambie amplitud entre qubits sin alterar el total de 1s.

La puerta **XY** hace exactamente eso: intercambia $|01\rangle \leftrightarrow |10\rangle$ (mueve un activo comprado de un qubit a otro):

$$U_{XY}(\beta) = e^{-i\beta(X_iX_j + Y_iY_j)} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & \cos 2\beta & -i\sin 2\beta & 0 \\ 0 & -i\sin 2\beta & \cos 2\beta & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

Aplicado en topología de anillo (pares $(1,2), (2,3), \ldots, (N,1)$), el mixer XY explora todo el subespacio de K activos.

Con estas dos modificaciones, el espacio de búsqueda efectivo se reduce de 1024 a 210 estados, y la probabilidad en el estado óptimo aumenta drásticamente.

In [ ]:
# ─── Datos del caso realista ──────────────────────────────────────────────────
N, K = 10, 6

μ = [11.5, 10.2, 9.0, 8.5, 7.5, 6.0, 5.5, 4.0, 2.5, 1.8]

cargas_factoriales = [
    2.5   2.0  -0.5;   # 1. Tech A
    2.2   1.8  -0.4;   # 2. Tech B
    2.0   1.5  -0.3;   # 3. Tech C
    1.8   0.0  -0.2;   # 4. Consumo Cíclico A
    1.6   0.0  -0.1;   # 5. Consumo Cíclico B
    1.0  -0.2   0.5;   # 6. Salud
    0.9  -0.2   0.6;   # 7. Consumo Básico
    0.6  -0.5   1.0;   # 8. Utilities
   -0.2  -0.5   1.5;   # 9. Bono Corporativo
   -0.5  -0.8   1.8    # 10. Bono del Tesoro
]

sigma = cargas_factoriales * cargas_factoriales'
riesgo_propio = [2.0, 1.8, 1.5, 1.5, 1.2, 1.0, 0.8, 0.6, 0.3, 0.1]
for i in 1:N; sigma[i,i] += riesgo_propio[i]; end
sigma = round.(sigma, digits=2)

# Sin término de penalización λ: con restricción dura ya no lo necesitamos
Q = construir_Q(μ, sigma, K; γ=1.0, λ=0.0)

println("Datos cargados: N=$N activos, K=$K en cartera")
println("Subespacio válido: C($N,$K) = ", binomial(N, K), " estados de ", 2^N)

### Preparación del Estado de Dicke

Usamos la preparación recursiva de Bartschi-Eidenbenz. La idea es que si tenemos $|D_k^n\rangle$ podemos construir $|D_k^{n+1}\rangle$ y $|D_{k-1}^{n}\rangle$ mediante rotaciones $R_y$ controladas y puertas CNOT.

La rotación clave en cada paso tiene ángulo:
$$\theta_{n,k} = 2\arccos\left(\sqrt{\frac{k}{n}}\right)$$

que distribuye exactamente la fracción correcta de amplitud hacia el qubit recién añadido.

In [ ]:
# ─── Preparación del Estado de Dicke |D_K^N> ─────────────────────────────────
# Algoritmo de Bartschi-Eidenbenz (2019): O(NK) puertas
function prep_dicke(N, K)
    circ = chain(N)
    # Paso 1: activar los primeros K qubits → |11...10...0>
    for i in 1:K
        push!(circ, put(N, i => X))
    end
    # Paso 2: difundir la amplitud hacia los qubits restantes
    # mediante rotaciones Ry controladas que conservan el peso de Hamming
    for i in 1:(N-1)
        for j in min(i, K):-1:max(1, i-N+K+1)
            θ = 2acos(sqrt(j / i))
            # Ry(θ) controlada por el qubit i condicionado a tener j excitaciones
            # Implementado como Ry controlada (CRy)
            push!(circ, control(N, i, (i+1) => Ry(θ)))
            push!(circ, cnot(N, i+1, i))
        end
    end
    return circ
end

# Verificación: el estado tiene exactamente las probabilidades correctas
reg_test = zero_state(N)
apply!(reg_test, prep_dicke(N, K))
probs_test = abs2.(statevec(reg_test))

# Contamos cuánta probabilidad está en estados con exactamente K activos
prob_valida = sum(probs_test[i+1] for i in 0:(2^N-1)
                  if count_ones(i) == K)
@printf("Probabilidad total en subespacio K=%d: %.4f (ideal: 1.0000)\n", K, prob_valida)

### Circuito QAOA con Mixer XY

El **Hamiltoniano de mezcla XY** en topología de anillo es:
$$H_{XY} = \sum_{i=1}^{N} (X_i X_{i+1} + Y_i Y_{i+1})$$

donde los índices son módulo N. Cada término conserva el número de excitaciones porque solo actúa sobre los subespacios $|01\rangle$ y $|10\rangle$ de cada par.

La implementación circuital de $e^{-i\beta(X_iX_j + Y_iY_j)}$ para un par $(i,j)$ es:

$$\text{CNOT}(j\to i) \cdot R_x(2\beta) \otimes R_z(2\beta) \cdot \text{CNOT}(j\to i)$$

Esto produce exactamente la matriz XY de 2×2 que mezcla $|01\rangle \leftrightarrow |10\rangle$.

In [ ]:
# ─── Puerta XY paramétrica ────────────────────────────────────────────────────
function xy_gate(β)
    c, s = cos(2β), sin(2β)
    ComplexF64[1 0 0 0;
               0 c -im*s 0;
               0 -im*s c 0;
               0 0 0 1] |> matblock
end

# ─── Capa QAOA con mixer XY ───────────────────────────────────────────────────
function capa_QAOA_xy(N, gamma, beta, h_ising, J_ising, pares)
    capa = chain(N)

    # Hamiltoniano de Coste H_C (idéntico al QAOA estándar)
    for i in 1:N
        push!(capa, put(N, i => Rz(2gamma * h_ising[i])))
    end
    for (i, j) in pares
        push!(capa, cnot(N, i, j))
        push!(capa, put(N, j => Rz(2gamma * J_ising[i,j])))
        push!(capa, cnot(N, i, j))
    end

    # Hamiltoniano de Mezcla XY (anillo: conserva el número de activos K)
    for i in 1:N
        j = mod1(i + 1, N)
        push!(capa, put(N, (i, j) => xy_gate(beta)))
    end

    return capa
end

# ─── Circuito completo QAOA-XY ────────────────────────────────────────────────
function circuito_QAOA_xy(N, K, p, gammas, betas, h_ising, J_ising, pares)
    circ = chain(N)
    # Estado inicial: Estado de Dicke (exactamente K activos, todos válidos)
    push!(circ, prep_dicke(N, K))
    # p capas QAOA con mixer XY
    for it in 1:p
        push!(circ, capa_QAOA_xy(N, gammas[it], betas[it], h_ising, J_ising, pares))
    end
    return circ
end;

## Optimización del Caso Realista

Aplicamos las tres mejoras de rendimiento al bucle de optimización:

| Técnica | Descripción | Impacto |
|---|---|---|
| **Circuito paramétrico** | La estructura del circuito se construye una vez; `dispatch!()` actualiza solo los ángulos | Evita ~100k reconstrucciones de `chain()` |
| **Parameter Shift Rule** | Gradientes exactos: $\frac{\partial E}{\partial\theta_i} = \frac{E(\theta_i+\pi/2) - E(\theta_i-\pi/2)}{2}$ | Sustituye diferencias finitas (2p evals extra/iter) |
| **Warm-start progresivo** | Optimizar p=1→12, interpolando la solución anterior | Reduce reinicios de 50 a 8 en el nivel final |

### Parameter Shift Rule

Para cualquier parámetro $\theta_i$ de una puerta de rotación $e^{-i\theta_i G}$ (donde $G$ tiene espectro $\{\pm\frac{1}{2}\}$):

$$\frac{\partial E}{\partial\theta_i} = \frac{1}{2}\left[E\left(\theta_i + \frac{\pi}{2}\right) - E\left(\theta_i - \frac{\pi}{2}\right)\right]$$

Este resultado es **exacto** (no una aproximación) y es compatible con hardware cuántico real.

In [ ]:
# ─── Precomputación (una sola vez, fuera del hot-path) ────────────────────────

# Coeficientes de Ising: h_i y J_ij cacheados
h_ising = [-Q[i,i]/2 - sum(Q[i,j]/2 for j in 1:N if j≠i) for i in 1:N]
J_ising = [(i<j) ? Q[i,j]/2 : 0.0 for i in 1:N, j in 1:N]

# Lista fija de pares con covarianza no nula
pares_acoplados = [(i,j) for i in 1:N for j in (i+1):N if abs(Q[i,j]) > 1e-3]

# Energías QUBO para los 2^N estados (precalculadas una sola vez)
# Solo sobre los estados válidos (K activos) para eficiencia
E_prec = [dot(digits(s, base=2, pad=N), Q * digits(s, base=2, pad=N))
          for s in 0:(2^N-1)]

println("Pares acoplados (covarianza > 0): ", length(pares_acoplados))
println("Energías precalculadas: ", length(E_prec), " estados")

In [ ]:
# ─── Circuito paramétrico: construir estructura una sola vez ──────────────────
# En lugar de reconstruir chain() en cada evaluación de perdida(),
# construimos la estructura con ángulos=0 y solo actualizamos los valores.
function build_qaoa_xy(N, K, p)
    circ = chain(N)
    push!(circ, prep_dicke(N, K))        # estado inicial fijo
    for _ in 1:p
        for i in 1:N                      # H_C lineal
            push!(circ, put(N, i => Rz(0.0)))
        end
        for (i,j) in pares_acoplados      # H_C ZZ
            push!(circ, cnot(N, i, j))
            push!(circ, put(N, j => Rz(0.0)))
            push!(circ, cnot(N, i, j))
        end
        for i in 1:N                      # H_M mixer XY
            j = mod1(i+1, N)
            push!(circ, put(N, (i,j) => xy_gate(0.0)))
        end
    end
    return circ
end

# Convierte (γ⃗, β⃗) → vector plano de ángulos en el mismo orden que parameters(circ)
function to_angles_xy(gammas, betas, p)
    v = Float64[]
    for k in 1:p
        for i in 1:N         push!(v, 2gammas[k] * h_ising[i]) end
        for (i,j) in pares_acoplados  push!(v, 2gammas[k] * J_ising[i,j]) end
        for _ in 1:N         push!(v, betas[k]) end  # xy_gate usa β directamente
    end
    return v
end

println("Estructura del circuito paramétrico lista para p=12")

In [ ]:
# ─── Función de pérdida + gradiente exacto (Parameter Shift Rule) ─────────────
function make_funciones_xy(p)
    circ = build_qaoa_xy(N, K, p)   # estructura construida UNA sola vez
    reg  = zero_state(N)             # búfer reutilizable (sin GC en hot-path)

    function perdida(θ)
        dispatch!(circ, to_angles_xy(θ[1:p], θ[p+1:end], p))
        # Reset in-place del registro (evita zero_state() en cada iter)
        fill!(reg.state, 0.0 + 0.0im)
        reg.state[1] = 1.0 + 0.0im
        apply!(reg, circ)
        return real(dot(abs2.(statevec(reg)), E_prec))
    end

    # Gradiente exacto con Parameter Shift Rule
    # ∂E/∂θᵢ = [E(θᵢ + π/2) − E(θᵢ − π/2)] / 2
    function gradiente!(G, θ)
        for i in eachindex(θ)
            θ₊, θ₋ = copy(θ), copy(θ)
            θ₊[i] += π/2; θ₋[i] -= π/2
            G[i] = (perdida(θ₊) - perdida(θ₋)) / 2
        end
    end

    return perdida, gradiente!
end

println("Funciones de pérdida y gradiente PSR listas")

In [ ]:
# ─── Warm-start progresivo: p = 1 → p_max ────────────────────────────────────
# En lugar de optimizar p=12 directamente (espacio 24-dimensional difícil),
# resolvemos p=1, interpolamos la solución a p=2, y así sucesivamente.
# Reduce los reinicios necesarios de 50 a ~8 en el nivel final.
function interpolar_params(θ_prev, p_nuevo)
    p_viejo = length(θ_prev) ÷ 2
    t_old   = LinRange(0.0, 1.0, p_viejo) |> collect
    t_new   = LinRange(0.0, 1.0, p_nuevo)
    function li(vals, t)
        k = clamp(searchsortedlast(t_old, t), 1, p_viejo-1)
        α = (t - t_old[k]) / (t_old[k+1] - t_old[k])
        return vals[k] * (1-α) + vals[k+1] * α
    end
    return vcat([li(θ_prev[1:p_viejo], t) for t in t_new],
                [li(θ_prev[p_viejo+1:end], t) for t in t_new])
end

println("Función de warm-start lista")

In [ ]:
# ─── Bucle principal de optimización ─────────────────────────────────────────
p_max = 12

println("╔══════════════════════════════════════════════════════╗")
println("║  QAOA-XY Optimizado  ·  N=$N activos, K=$K en cartera ║")
println("╚══════════════════════════════════════════════════════╝")
println()

θ_prev  = vcat([0.5], [0.5])  # punto de arranque para p=1
θ_mejor = zeros(2p_max)
E_mejor = Inf

for p_cur in 1:p_max
    perdida, gradiente! = make_funciones_xy(p_cur)
    θ_warm = interpolar_params(θ_prev, p_cur)

    E_mejor_p = Inf
    θ_mejor_p = copy(θ_warm)

    # Más intentos y más iteraciones solo en el nivel final
    n_intentos = (p_cur == p_max) ? 8 : 2
    max_iters  = (p_cur == p_max) ? 1500 : 350

    for intento in 1:n_intentos
        # Intento 1: warm-start interpolado; resto: arranques aleatorios heurísticos
        θ₀ = (intento == 1) ? θ_warm :
             vcat(sort(rand(p_cur)) .* 2π, sort(rand(p_cur), rev=true))

        res = optimize(perdida, gradiente!, θ₀, LBFGS(),
                       Optim.Options(iterations=max_iters, g_tol=1e-5,
                                     show_trace=false))

        if Optim.minimum(res) < E_mejor_p
            E_mejor_p = Optim.minimum(res)
            θ_mejor_p = Optim.minimizer(res)
        end
    end

    @printf("  p = %2d  →  E_min = %+.6f\n", p_cur, E_mejor_p)
    θ_prev = θ_mejor_p

    if p_cur == p_max
        E_mejor = E_mejor_p
        θ_mejor = θ_mejor_p
    end
end

In [ ]:
# ─── Medición final ───────────────────────────────────────────────────────────
circ_opt = build_qaoa_xy(N, K, p_max)
dispatch!(circ_opt, to_angles_xy(θ_mejor[1:p_max], θ_mejor[p_max+1:end], p_max))

reg_final = zero_state(N)
apply!(reg_final, circ_opt)
probs = abs2.(statevec(reg_final))

# Ordenar por probabilidad y filtrar solo carteras con exactamente K activos
carteras = [(digits(i, base=2, pad=N), probs[i+1]) for i in 0:(2^N-1)
             if count_ones(i) == K]
sort!(carteras, by=x->x[2], rev=true)

println()
println("╔═══════════════════════════════════════════════════════╗")
println("║             TOP 5 CARTERAS (N=10, K=6)                ║")
println("╚═══════════════════════════════════════════════════════╝")
for k in 1:5
    bits, pk = carteras[k]
    activos  = findall(==(1), bits)
    ret_esp  = sum(μ[i] for i in activos)
    riesgo   = sqrt(activos' * sigma[activos, activos] * activos)
    @printf("  %d.  %5.2f%%  │  Retorno: %5.1f%%  │  Riesgo: %5.2f  │  Activos: %s\n",
            k, pk*100, ret_esp, riesgo, join(activos, ", "))
end

# Probabilidad total en el subespacio válido (debe ser ≈ 1.0 con mixer XY)
prob_subespacio = sum(probs[i+1] for i in 0:(2^N-1) if count_ones(i) == K)
@printf("\n  Probabilidad total en subespacio K=%d: %.4f\n", K, prob_subespacio)
@printf("  Energía mínima encontrada: %+.6f\n", E_mejor)